In [ ]:
import matplotlib as mpl
from scipy.optimize import minimize
%matplotlib inline
# Set the font to Times New Roman or a similar serif font
mpl.rcParams['font.family'] = 'serif'
mpl.rcParams['font.serif'] = 'Times New Roman'
# Adjust the text sizes
mpl.rcParams['axes.labelsize'] = 10  # For X and Y axis labels
mpl.rcParams['axes.titlesize'] = 10  # For the plot title
mpl.rcParams['legend.fontsize'] = 10 # For the legend


from utils import *


In [ ]:
phase_01_1em3 = 1.58437266
phase_12_1em3 = -1.890425
phase_01_0 = 1.60195078125
phase_12_0 = -1.87626484375

zero = qutip.basis(2, 0)
one = qutip.basis(2, 1)

gate = qutip.qip.operations.phasegate(phase)

Paulis = [qutip.sigmax(),qutip.sigmay(),qutip.sigmaz()]
    
colors = [(0.9, 0.9, 0.4), (0.5*0.5, 0.7*0.5, 0.5*0.5), (0.7, 0.6, 0.7)]
linestyles = [(0,(3,1,1,1)),'-',(0,(5,2,5,2))]
fig = plt.figure(figsize=((3+3/8)*1.4, 
                            (3+3/8)*1.7))
gs = gridspec.GridSpec(1,2, width_ratios=[1], height_ratios=[1,1],wspace=1.0,hspace=0.2)



# two rows for different qubit
for i, filename in enumerate([
            'pickles/mesolve_01_pauli_dm.pkl',
            'pickles/mesolve_12_pauli_dm.pkl'
        ]):
    with open(filename, 'rb') as file:
        results_combined = pickle.load(file)
    
    # different kappa will be plotted in the same subplots
    for j, (kappa, results) in enumerate(zip([1e-3,0],[results_combined[0:4],results_combined[4:]])):
        ax = plt.subplot(gs[i,0])
        for P, Pauli in zip(['X','Y','Z'],Paulis):
            for a,b, final_states in zip(
                [zero,zero,one,one],
                [zero,one,zero,one],
                []
            )


            num_initial_states = len(results)
            num_time_steps = len(results[0].times)
            # distinguish the results for different kappa
            infidelity = []
            phase = []
            for idx in tqdm(range(num_time_steps), desc='Processing'):
                dms = [result.states[idx] for result in results]
                def objective_function(phase):
                    return calc_average_fidelity_with_phase(phase[0], dms, states_ideal)
                initial_phase = [0.0]
                bounds = [(0, 2 * 3.141592653589793)]
                opt_result = minimize(objective_function, initial_phase,method="COBYLA")
                infidelity.append(opt_result.fun)
                phase.append(opt_result.x)

            color = 'tab:red'
            ax.plot(results[0].times,phase,label = 'phase',color = color)
            ax.set_ylabel('Phase', color=color)
            ax.set_xlabel("t (ns)")
            ax.grid(which='major', linestyle='-', linewidth='0.5', color='black')
            # ax1.minorticks_on()
            # ax1.grid(which='minor', linestyle='--', linewidth='0.5', color='gray')
            ax.tick_params(axis='y', labelcolor=color)
            ax.legend(loc='upper left')

        ax2 = ax1.twinx()
        color = 'tab:blue'
        ax2.set_ylabel('Infidelity', color=color)
        ax2.tick_params(axis='y', labelcolor=color)
        ax2.plot(results[0].times,infidelity,label = 'Infidelity',color = color)
        ax2.legend(loc='lower right')
        ax2.set_yscale('log')
fig.tight_layout()  
plt.show()